<a href="https://colab.research.google.com/github/varshini-cit/DAA-LAB/blob/main/8A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# Travelling Salesman Problem using Branch and Bound
# ============================================================
#
# Find the minimum cost Hamiltonian cycle for a 5-city TSP.
#
# Branch and Bound uses:
#     - Cost matrix reduction
#     - Lower-bound estimation
#     - Priority queue
#
# ============================================================

import heapq


# ------------------------------------------------------------
# Cost Matrix
# ------------------------------------------------------------
# 0 means that the city cannot travel to itself.
#
# You can replace this matrix with the matrix given by
# your faculty/assignment.

COST_MATRIX = [
    [0, 20, 30, 10, 11],
    [15, 0, 16, 4, 2],
    [3, 5, 0, 2, 4],
    [19, 6, 18, 0, 3],
    [16, 4, 7, 16, 0]
]

N = len(COST_MATRIX)

INF = float("inf")


# ------------------------------------------------------------
# Reduce Cost Matrix
# ------------------------------------------------------------

def reduce_matrix(matrix):

    n = len(matrix)

    reduced_matrix = [
        row[:] for row in matrix
    ]

    reduction_cost = 0

    # --------------------------------------------------------
    # Row Reduction
    # --------------------------------------------------------

    for i in range(n):

        row_min = min(reduced_matrix[i])

        if row_min != INF and row_min != 0:

            reduction_cost += row_min

            for j in range(n):

                if reduced_matrix[i][j] != INF:
                    reduced_matrix[i][j] -= row_min

    # --------------------------------------------------------
    # Column Reduction
    # --------------------------------------------------------

    for j in range(n):

        column_min = INF

        for i in range(n):

            if reduced_matrix[i][j] < column_min:
                column_min = reduced_matrix[i][j]

        if column_min != INF and column_min != 0:

            reduction_cost += column_min

            for i in range(n):

                if reduced_matrix[i][j] != INF:
                    reduced_matrix[i][j] -= column_min

    return reduced_matrix, reduction_cost


# ------------------------------------------------------------
# Create Child Matrix
# ------------------------------------------------------------

def create_child_matrix(matrix, from_city, to_city):

    n = len(matrix)

    child_matrix = [
        row[:] for row in matrix
    ]

    # Make the entire row of from_city unavailable
    for j in range(n):
        child_matrix[from_city][j] = INF

    # Make the entire column of to_city unavailable
    for i in range(n):
        child_matrix[i][to_city] = INF

    # Prevent immediate return to from_city
    child_matrix[to_city][from_city] = INF

    return child_matrix


# ------------------------------------------------------------
# Branch and Bound TSP
# ------------------------------------------------------------

def tsp_branch_and_bound(cost_matrix):

    n = len(cost_matrix)

    # Initial matrix reduction
    reduced_matrix, initial_cost = reduce_matrix(cost_matrix)

    # Priority queue:
    # (lower_bound, current_city, path, matrix)
    priority_queue = []

    heapq.heappush(
        priority_queue,
        (
            initial_cost,
            0,
            [0],
            reduced_matrix
        )
    )

    best_cost = INF
    best_path = []

    nodes_explored = 0

    while priority_queue:

        lower_bound, current_city, path, matrix = heapq.heappop(
            priority_queue
        )

        nodes_explored += 1

        # Prune this node if its lower bound is already
        # greater than or equal to the best solution found.
        if lower_bound >= best_cost:
            continue

        # ----------------------------------------------------
        # Complete Tour
        # ----------------------------------------------------

        if len(path) == n:

            last_city = path[-1]

            return_cost = cost_matrix[last_city][0]

            if return_cost != 0 and return_cost != INF:

                total_cost = lower_bound + return_cost

                if total_cost < best_cost:

                    best_cost = total_cost
                    best_path = path + [0]

            continue

        # ----------------------------------------------------
        # Branch to Unvisited Cities
        # ----------------------------------------------------

        for next_city in range(n):

            if next_city in path:
                continue

            edge_cost = matrix[current_city][next_city]

            if edge_cost == INF:
                continue

            # Create child matrix
            child_matrix = create_child_matrix(
                matrix,
                current_city,
                next_city
            )

            # Reduce child matrix
            child_matrix, reduction_cost = reduce_matrix(
                child_matrix
            )

            # Calculate child's lower bound
            child_bound = (
                lower_bound
                + edge_cost
                + reduction_cost
            )

            # Only add promising branches
            if child_bound < best_cost:

                heapq.heappush(
                    priority_queue,
                    (
                        child_bound,
                        next_city,
                        path + [next_city],
                        child_matrix
                    )
                )

    return best_cost, best_path, nodes_explored


# ------------------------------------------------------------
# Display Cost Matrix
# ------------------------------------------------------------

def display_matrix(matrix):

    print("\nCost Matrix:")

    print("      ", end="")

    for i in range(N):
        print(f"{i:>7}", end="")

    print()

    print("-" * (7 * (N + 1)))

    for i in range(N):

        print(f"{i:>3}   ", end="")

        for j in range(N):

            print(f"{matrix[i][j]:>7}", end="")

        print()


# ------------------------------------------------------------
# Main Function
# ------------------------------------------------------------

def main():

    print("=" * 65)
    print("TRAVELLING SALESMAN PROBLEM")
    print("BRANCH AND BOUND")
    print("=" * 65)

    print(f"\nNumber of cities: {N}")
    print("Starting city: 0")

    display_matrix(COST_MATRIX)

    # Solve TSP
    best_cost, best_path, nodes_explored = tsp_branch_and_bound(
        COST_MATRIX
    )

    # --------------------------------------------------------
    # Display Result
    # --------------------------------------------------------

    print("\n" + "=" * 65)
    print("OPTIMAL SOLUTION")
    print("=" * 65)

    if best_path:

        tour = " -> ".join(
            map(str, best_path)
        )

        print(f"\nOptimal Tour: {tour}")
        print(f"Minimum Tour Cost: {best_cost}")

    else:

        print("\nNo Hamiltonian cycle exists.")

    print(f"\nNodes explored: {nodes_explored}")

    # --------------------------------------------------------
    # Complexity
    # --------------------------------------------------------

    print("\n" + "=" * 65)
    print("COMPLEXITY ANALYSIS")
    print("=" * 65)

    print("""
Worst-case Time Complexity:
    O(n² × 2ⁿ)

Space Complexity:
    O(n × 2ⁿ)

Branch and Bound reduces the practical search space by
pruning branches whose lower bound is already worse than
the best solution found so far.
""")


# ------------------------------------------------------------
# Program Entry Point
# ------------------------------------------------------------

if __name__ == "__main__":
    main()

TRAVELLING SALESMAN PROBLEM
BRANCH AND BOUND

Number of cities: 5
Starting city: 0

Cost Matrix:
            0      1      2      3      4
------------------------------------------
  0         0     20     30     10     11
  1        15      0     16      4      2
  2         3      5      0      2      4
  3        19      6     18      0      3
  4        16      4      7     16      0

OPTIMAL SOLUTION

Optimal Tour: 0 -> 3 -> 1 -> 4 -> 2 -> 0
Minimum Tour Cost: 31

Nodes explored: 24

COMPLEXITY ANALYSIS

Worst-case Time Complexity:
    O(n² × 2ⁿ)

Space Complexity:
    O(n × 2ⁿ)

Branch and Bound reduces the practical search space by
pruning branches whose lower bound is already worse than
the best solution found so far.

